# GraphSAGE Recommendation – REES46
Phương pháp: **GraphSAGE** (PyTorch Geometric) – học embedding trên đồ thị user-item  
Metric: **Recall@10/20/50** và **NDCG@10/20/50** trên val & test set

**So sánh vs SVD baseline:**
- SVD: matrix factorization đơn giản, không học cấu trúc đồ thị
- GraphSAGE: học embedding bằng cách aggregate thông tin từ neighbors → hiểu cấu trúc quan hệ user-item sâu hơn

## 0. Cài đặt dependencies

In [1]:
!git clone https://huggingface.co/datasets/nguyenmaiductrong/rees46-processed-data

Cloning into 'rees46-processed-data'...
remote: Enumerating objects: 25, done.
remote: Total 25 (delta 0), reused 0 (delta 0), pack-reused 25 (from 1)
Receiving objects: 100% (25/25), 1.77 MiB | 3.56 MiB/s, done.
Filtering content: 100% (13/13), 675.87 MiB | 438.03 MiB/s, done.


In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Cài PyG phù hợp với version torch hiện tại
import subprocess, sys
torch_ver = torch.__version__.split('+')[0]  # e.g. '2.1.0'
cuda_ver  = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f"\nĐang cài PyTorch Geometric cho torch={torch_ver}, {cuda_ver}...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch-geometric",
    "--find-links", f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html"
], check=True)
print("Cài xong!")

PyTorch version: 2.10.0+cu128
CUDA available: True

Đang cài PyTorch Geometric cho torch=2.10.0, cu121...
Cài xong!


In [3]:
import numpy as np
import json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import HeteroData
from scipy.sparse import coo_matrix, csr_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

data_dir = "/content/rees46-processed-data/temporal"

Device: cuda


## 1. Load dữ liệu

In [4]:
view_src     = np.load(f"{data_dir}/view_train_src.npy")
view_dst     = np.load(f"{data_dir}/view_train_dst.npy")
cart_src     = np.load(f"{data_dir}/cart_train_src.npy")
cart_dst     = np.load(f"{data_dir}/cart_train_dst.npy")
purchase_src = np.load(f"{data_dir}/purchase_train_src.npy")
purchase_dst = np.load(f"{data_dir}/purchase_train_dst.npy")

val_user     = np.load(f"{data_dir}/val_user_idx.npy")
val_product  = np.load(f"{data_dir}/val_product_idx.npy")
test_user    = np.load(f"{data_dir}/test_user_idx.npy")
test_product = np.load(f"{data_dir}/test_product_idx.npy")

with open(f"{data_dir}/node_counts.json") as f:
    node_counts = json.load(f)

num_users = node_counts['user']
num_items = node_counts['product']
print(f"Users: {num_users:,} | Items: {num_items:,}")
print(f"Val: {len(val_user):,} | Test: {len(test_user):,}")

Users: 267,973 | Items: 60,798
Val: 415,195 | Test: 188,112


## 2. Xây dựng đồ thị user-item

Bipartite graph: mỗi cạnh là tương tác user→item, có trọng số theo loại:  
view=1, cart=2, purchase=3

Đồ thị là **undirected** (thêm cả chiều ngược item→user) để GraphSAGE aggregate từ cả 2 phía.

In [5]:
# Gộp tất cả edges + trọng số
all_src = np.concatenate([view_src, cart_src, purchase_src])
all_dst = np.concatenate([view_dst, cart_dst, purchase_dst])
all_w   = np.concatenate([
    np.ones(len(view_src)),
    np.full(len(cart_src), 2.0),
    np.full(len(purchase_src), 3.0)
])

# Deduplicate: nếu 1 user tương tác cùng 1 item nhiều loại → cộng dồn weight
R = csr_matrix(
    coo_matrix((all_w, (all_src, all_dst)), shape=(num_users, num_items))
)
coo = R.tocoo()
u_idx = coo.row.astype(np.int64)
i_idx = coo.col.astype(np.int64)
weights = coo.data.astype(np.float32)

print(f"Edges (unique user-item pairs): {len(u_idx):,}")
print(f"R density: {len(u_idx)/(num_users*num_items)*100:.4f}%")

# Edge index cho graph: user nodes=[0..num_users-1], item nodes=[0..num_items-1]
# Undirected: thêm cả chiều ngược
edge_index_ui = torch.tensor([u_idx, i_idx], dtype=torch.long)  # user→item
edge_index_iu = torch.tensor([i_idx, u_idx], dtype=torch.long)  # item→user
edge_weight   = torch.tensor(weights, dtype=torch.float32)

print(f"edge_index_ui: {edge_index_ui.shape}")

Edges (unique user-item pairs): 15,045,001
R density: 0.0923%
edge_index_ui: torch.Size([2, 15045001])


/tmp/ipykernel_2047/3054315631.py:24: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index_ui = torch.tensor([u_idx, i_idx], dtype=torch.long)  # user→item


## 3. Định nghĩa mô hình GraphSAGE

**Kiến trúc:**
- Node features khởi tạo ngẫu nhiên (learnable embeddings)
- 2 lớp SAGEConv: aggregate thông tin từ neighbors
- Score = dot product giữa user_emb và item_emb
- Loss: BPR (Bayesian Personalized Ranking) – positive items nên score cao hơn negative

In [6]:
class GraphSAGE_RecSys(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.emb_dim   = emb_dim

        # Learnable embeddings cho user và item
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

        # SAGEConv layers – hoạt động trên homogeneous graph
        # Ta sẽ dùng trên bipartite graph bằng cách xây node feature chung
        self.sage1 = SAGEConv(emb_dim, hidden_dim)
        self.sage2 = SAGEConv(hidden_dim, emb_dim)

        self.dropout = nn.Dropout(dropout)
        self.norm1   = nn.LayerNorm(hidden_dim)
        self.norm2   = nn.LayerNorm(emb_dim)

    def forward(self, edge_index_ui, edge_index_iu):
        """
        edge_index_ui: [2, E] – user→item edges (user_id, item_id)
        edge_index_iu: [2, E] – item→user edges (item_id, user_id)
        Returns: user_out [num_users, emb_dim], item_out [num_items, emb_dim]
        """
        u0 = self.user_emb.weight  # (num_users, emb_dim)
        i0 = self.item_emb.weight  # (num_items, emb_dim)

        # === Layer 1 ===
        # user ← items: aggregate items (i0) to users (u0) using item->user edges (edge_index_iu)
        u1 = self.sage1(
            (i0, u0),              # (src_feat, dst_feat)
            edge_index_iu          # Correct: [item_id, user_id]
        )  # shape: (num_users, hidden_dim)
        u1 = self.norm1(F.relu(u1))
        u1 = self.dropout(u1)

        # item ← users: aggregate users (u0) to items (i0) using user->item edges (edge_index_ui)
        i1 = self.sage1(
            (u0, i0),
            edge_index_ui          # Correct: [user_id, item_id]
        )  # shape: (num_items, hidden_dim)
        i1 = self.norm1(F.relu(i1))
        i1 = self.dropout(i1)

        # === Layer 2 ===
        # user ← items: aggregate items (i1) to users (u1) using item->user edges (edge_index_iu)
        u2 = self.sage2((i1, u1), edge_index_iu)
        u2 = self.norm2(u2)

        # item ← users: aggregate users (u1) to items (i1) using user->item edges (edge_index_ui)
        i2 = self.sage2((u1, i1), edge_index_ui)
        i2 = self.norm2(i2)

        # Residual connection với embedding ban đầu
        user_out = u2 + u0
        item_out = i2 + i0

        return user_out, item_out

    def bpr_loss(self, user_emb, item_emb, users, pos_items, neg_items, l2_reg=1e-4):
        """
        BPR Loss: score(user, pos) > score(user, neg)
        """
        u   = user_emb[users]       # (batch, emb)
        pos = item_emb[pos_items]   # (batch, emb)
        neg = item_emb[neg_items]   # (batch, emb)

        pos_score = (u * pos).sum(dim=1)  # (batch,)
        neg_score = (u * neg).sum(dim=1)

        loss = -F.logsigmoid(pos_score - neg_score).mean()

        # L2 regularization trên embeddings gốc (tránh overfit)
        l2 = (self.user_emb.weight[users].norm(2).pow(2)
             + self.item_emb.weight[pos_items].norm(2).pow(2)
             + self.item_emb.weight[neg_items].norm(2).pow(2)) / len(users)

        return loss + l2_reg * l2


print("Model class defined.")

Model class defined.


## 4. Chuẩn bị training

**Negative sampling:** với mỗi (user, pos_item), sample ngẫu nhiên 1 item mà user chưa tương tác.

In [7]:
# Tạo set items đã tương tác của mỗi user (để negative sampling)
from collections import defaultdict

print("Đang build user interaction sets...")
t0 = time.time()
user_items = defaultdict(set)
for u, i in zip(u_idx, i_idx):
    user_items[u].add(i)
print(f"Xong! {time.time()-t0:.1f}s")

# Danh sách (user, pos_item) để train
train_pairs = list(zip(u_idx.tolist(), i_idx.tolist()))
print(f"Train pairs: {len(train_pairs):,}")

def sample_negatives(users, user_items, num_items, n_neg=1):
    """Sample negative items cho mỗi user."""
    neg_items = []
    for u in users:
        interacted = user_items[u]
        while True:
            neg = np.random.randint(0, num_items)
            if neg not in interacted:
                neg_items.append(neg)
                break
    return neg_items

print("Negative sampling function ready.")

Đang build user interaction sets...
Xong! 3.2s
Train pairs: 15,045,001
Negative sampling function ready.


## 5. Training

In [8]:
# ── Hyperparameters ──────────────────────────────────────────
EMB_DIM    = 64        # kích thước embedding (same as SVD K=64)
HIDDEN_DIM = 128
LR         = 1e-3
L2_REG     = 1e-4
BATCH_SIZE = 4096
NUM_EPOCHS = 20        # tăng lên 50+ để kết quả tốt hơn
EVAL_EVERY = 5         # đánh giá val mỗi N epoch
# ─────────────────────────────────────────────────────────────

model = GraphSAGE_RecSys(
    num_users=num_users,
    num_items=num_items,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Đưa edge index lên GPU
ei_ui = edge_index_ui.to(DEVICE)
ei_iu = edge_index_iu.to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")
print(f"Device: {DEVICE}")
REFRESH_EVERY = 50      # recompute embeddings mỗi N batches để cân bằng speed vs memory


Model parameters: 21,074,688
Device: cuda


In [12]:
def train_epoch(model, optimizer, ei_ui, ei_iu, train_pairs, user_items, num_items, batch_size):
    model.train()
    np.random.shuffle(train_pairs)
    total_loss = 0.0
    n_batches  = 0

    # Full-graph forward ONCE per REFRESH_EVERY batches
    # Lấy embeddings + giữ gradient graph
    user_emb, item_emb = model(ei_ui, ei_iu)

    optimizer.zero_grad() # Zero gradients once per REFRESH_EVERY cycle

    for start in range(0, len(train_pairs), batch_size):
        batch = train_pairs[start:start + batch_size]
        users_b     = [p[0] for p in batch]
        pos_items_b = [p[1] for p in batch]
        neg_items_b = sample_negatives(users_b, user_items, num_items)

        u_t   = torch.tensor(users_b,      dtype=torch.long, device=DEVICE)
        pos_t = torch.tensor(pos_items_b,  dtype=torch.long, device=DEVICE)
        neg_t = torch.tensor(neg_items_b,  dtype=torch.long, device=DEVICE)

        # Chỉ tính loss trên sub-batch, backward qua embedding đã tính sẵn
        loss = model.bpr_loss(user_emb, item_emb, u_t, pos_t, neg_t, l2_reg=L2_REG)

        # Accumulate gradients. retain_graph=True if not the last batch in the cycle
        is_last_batch_in_cycle = (n_batches + 1) % REFRESH_EVERY == 0
        is_last_batch_of_epoch = (start + batch_size >= len(train_pairs))
        loss.backward(retain_graph= not (is_last_batch_in_cycle or is_last_batch_of_epoch))

        total_loss += loss.item()
        n_batches  += 1

        # Perform optimizer step and recompute embeddings at the end of each REFRESH_EVERY cycle
        if is_last_batch_in_cycle or is_last_batch_of_epoch:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step() # Update model parameters using accumulated gradients
            optimizer.zero_grad() # Clear gradients for the next cycle

            # Recompute embeddings after updating parameters
            # Ensure model is in train mode before forward pass
            model.train()
            user_emb, item_emb = model(ei_ui, ei_iu)

    return total_loss / n_batches


print("Training loop ready.")

Training loop ready.


In [13]:
print(f"Bắt đầu training {NUM_EPOCHS} epochs...")
print(f"Batch size: {BATCH_SIZE} | LR: {LR} | EMB_DIM: {EMB_DIM}")
print("=" * 60)

train_start = time.time()
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    loss = train_epoch(model, optimizer, ei_ui, ei_iu,
                       train_pairs, user_items, num_items, BATCH_SIZE)
    scheduler.step()
    elapsed = time.time() - t0

    history.append({'epoch': epoch, 'loss': loss})
    print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Loss: {loss:.4f} | LR: {scheduler.get_last_lr()[0]:.5f} | Time: {elapsed:.1f}s")

print(f"\nTổng thời gian training: {time.time()-train_start:.1f}s")
# REFRESH_EVERY = 50      # This line is redundant as it's defined globally before

Bắt đầu training 20 epochs...
Batch size: 4096 | LR: 0.001 | EMB_DIM: 64
Epoch   1/20 | Loss: 0.3535 | LR: 0.00100 | Time: 281.1s
Epoch   2/20 | Loss: 0.2039 | LR: 0.00100 | Time: 280.5s
Epoch   3/20 | Loss: 0.1617 | LR: 0.00100 | Time: 280.5s
Epoch   4/20 | Loss: 0.1376 | LR: 0.00100 | Time: 280.6s
Epoch   5/20 | Loss: 0.1236 | LR: 0.00100 | Time: 280.5s
Epoch   6/20 | Loss: 0.1120 | LR: 0.00100 | Time: 280.5s
Epoch   7/20 | Loss: 0.1031 | LR: 0.00100 | Time: 280.5s
Epoch   8/20 | Loss: 0.0958 | LR: 0.00100 | Time: 280.5s
Epoch   9/20 | Loss: 0.0901 | LR: 0.00100 | Time: 280.5s
Epoch  10/20 | Loss: 0.0855 | LR: 0.00050 | Time: 280.5s
Epoch  11/20 | Loss: 0.0812 | LR: 0.00050 | Time: 280.5s
Epoch  12/20 | Loss: 0.0785 | LR: 0.00050 | Time: 280.5s
Epoch  13/20 | Loss: 0.0760 | LR: 0.00050 | Time: 280.5s
Epoch  14/20 | Loss: 0.0736 | LR: 0.00050 | Time: 280.5s
Epoch  15/20 | Loss: 0.0717 | LR: 0.00050 | Time: 280.5s
Epoch  16/20 | Loss: 0.0697 | LR: 0.00050 | Time: 280.5s
Epoch  17/20 | 

## 6. Lấy embeddings cuối để đánh giá

In [14]:
print("Đang lấy final embeddings...")
model.eval()
with torch.no_grad():
    user_emb_final, item_emb_final = model(ei_ui, ei_iu)

# Chuyển sang numpy để dùng chung evaluation code với SVD
user_emb_np = user_emb_final.cpu().numpy()  # (num_users, EMB_DIM)
item_emb_np = item_emb_final.cpu().numpy()  # (num_items, EMB_DIM)

print(f"user_emb: {user_emb_np.shape} | item_emb: {item_emb_np.shape}")

Đang lấy final embeddings...
user_emb: (267973, 64) | item_emb: (60798, 64)


## 7. Đánh giá – Recall@K & NDCG@K

Giữ nguyên evaluation code từ SVD notebook để so sánh fair.

In [15]:
def recall_at_k(user_emb, item_emb, user_idx, true_items, k=20, batch_size=1024):
    hits = 0
    n = len(user_idx)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch  = user_idx[start:end]
        gt_batch = true_items[start:end]
        scores   = user_emb[u_batch] @ item_emb.T
        top_k    = np.argpartition(-scores, k, axis=1)[:, :k]
        for i, gt in enumerate(gt_batch):
            if gt in top_k[i]:
                hits += 1
    return hits / n


def ndcg_at_k(user_emb, item_emb, user_idx, true_items, k=20, batch_size=1024):
    total = 0.0
    n = len(user_idx)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch  = user_idx[start:end]
        gt_batch = true_items[start:end]
        scores    = user_emb[u_batch] @ item_emb.T
        top_k_idx = np.argsort(-scores, axis=1)[:, :k]
        for i, gt in enumerate(gt_batch):
            ranks = np.where(top_k_idx[i] == gt)[0]
            if len(ranks) > 0:
                total += 1.0 / np.log2(ranks[0] + 2)
    return total / n


def evaluate_all(user_emb, item_emb, user_idx, true_items, label="", batch_size=1024):
    r10 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=10,  batch_size=batch_size)
    r20 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=20,  batch_size=batch_size)
    r50 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=50,  batch_size=batch_size)
    n10 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=10,  batch_size=batch_size)
    n20 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=20,  batch_size=batch_size)
    n50 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=50,  batch_size=batch_size)
    hdr = f"  {label}  " if label else "  "
    print(f"\n{'='*52}")
    print(f"{hdr}Recall@10  : {r10:.4f}")
    print(f"{hdr}Recall@20  : {r20:.4f}")
    print(f"{hdr}Recall@50  : {r50:.4f}")
    print(f"{hdr}NDCG@10    : {n10:.4f}")
    print(f"{hdr}NDCG@20    : {n20:.4f}")
    print(f"{hdr}NDCG@50    : {n50:.4f}")
    print(f"{'='*52}")
    return dict(r10=r10, r20=r20, r50=r50, n10=n10, n20=n20, n50=n50)


print("Evaluation functions ready.")

Evaluation functions ready.


In [16]:
print("Đang đánh giá trên Val set...")
t0 = time.time()
val_metrics = evaluate_all(user_emb_np, item_emb_np, val_user, val_product, label="VAL")
print(f"  Thời gian: {time.time()-t0:.1f}s")

Đang đánh giá trên Val set...

  VAL  Recall@10  : 0.1666
  VAL  Recall@20  : 0.2276
  VAL  Recall@50  : 0.3227
  VAL  NDCG@10    : 0.0917
  VAL  NDCG@20    : 0.1071
  VAL  NDCG@50    : 0.1259
  Thời gian: 1081.3s


In [17]:
print("Đang đánh giá trên Test set...")
t0 = time.time()
test_metrics = evaluate_all(user_emb_np, item_emb_np, test_user, test_product, label="TEST")
print(f"  Thời gian: {time.time()-t0:.1f}s")

Đang đánh giá trên Test set...

  TEST  Recall@10  : 0.0823
  TEST  Recall@20  : 0.1216
  TEST  Recall@50  : 0.1827
  TEST  NDCG@10    : 0.0420
  TEST  NDCG@20    : 0.0520
  TEST  NDCG@50    : 0.0641
  Thời gian: 491.1s


## 8. So sánh vs SVD baseline

In [18]:
# SVD results từ notebook gốc
svd_val  = dict(r10=0.2214, r20=0.2634, r50=0.3202, n10=0.1445, n20=0.1551, n50=0.1664)
svd_test = dict(r10=0.1075, r20=0.1376, r50=0.1837, n10=0.0634, n20=0.0710, n50=0.0801)

print("\n" + "="*65)
print(f"{'Metric':<15} {'SVD Val':>10} {'SAGE Val':>10} {'SVD Test':>10} {'SAGE Test':>10}")
print("-"*65)
for k in ['r10', 'r20', 'r50', 'n10', 'n20', 'n50']:
    label = k.replace('r', 'Recall@').replace('n', 'NDCG@')
    sv  = svd_val[k]
    gv  = val_metrics[k]
    st  = svd_test[k]
    gt  = test_metrics[k]
    dv  = (gv - sv) / sv * 100
    dt  = (gt - st) / st * 100
    sign_v = '+' if dv >= 0 else ''
    sign_t = '+' if dt >= 0 else ''
    print(f"{label:<15} {sv:>10.4f} {gv:>9.4f}({sign_v}{dv:.1f}%) {st:>10.4f} {gt:>9.4f}({sign_t}{dt:.1f}%)")
print("="*65)
print("(+X%) = GraphSAGE tốt hơn SVD X%, (-X%) = kém hơn)")


Metric             SVD Val   SAGE Val   SVD Test  SAGE Test
-----------------------------------------------------------------
Recall@10           0.2214    0.1666(-24.8%)     0.1075    0.0823(-23.4%)
Recall@20           0.2634    0.2276(-13.6%)     0.1376    0.1216(-11.6%)
Recall@50           0.3202    0.3227(+0.8%)     0.1837    0.1827(-0.5%)
NDCG@10             0.1445    0.0917(-36.6%)     0.0634    0.0420(-33.7%)
NDCG@20             0.1551    0.1071(-31.0%)     0.0710    0.0520(-26.8%)
NDCG@50             0.1664    0.1259(-24.3%)     0.0801    0.0641(-20.0%)
(+X%) = GraphSAGE tốt hơn SVD X%, (-X%) = kém hơn)


## 9. Tips để cải thiện kết quả

Nếu GraphSAGE chưa beat SVD, thử:

1. **Tăng epochs**: `NUM_EPOCHS = 50` hoặc `100` – SVD converge ngay, GNN cần nhiều epoch hơn  
2. **Tăng EMB_DIM**: `128` hoặc `256`
3. **Mini-batch GraphSAGE** (NeighborSampler) thay vì full-graph forward – scale tốt hơn
4. **LightGCN** thay SAGEConv – simpler, thường tốt hơn cho collaborative filtering
5. **Điều chỉnh trọng số**: thay vì cộng dồn, thử normalize weight theo user degree
6. **Hard negative mining**: thay vì random neg, chọn neg score cao nhất